In [1]:
import hoda
import tensorly as tl

print(tl.get_backend())
%pip freeze | grep moabb

cupy
moabb==1.0.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation
tmin = 0
tmax=0.8
fmin=0.5
fmax = 16
sfreq = 32

paradigm = P300(resample=sfreq, tmin=tmin, tmax=tmax, fmin=fmin, fmax=fmax)
datasets = [
    #BI2012(),
    #BI2013a(),
    #BI2014a(),
    #BI2014b(),
    #BI2015a(),
    #BI2015b(),
    BNCI2014_008(),
    #BNCI2014_009(),
    #BNCI2015_003(),
    #Cattan2019_VR(),
    #EPFLP300(),
    #Huebner2017(),
    #Huebner2018(),
    #Lee2019_ERP(),
    #Sosulski2019(),
]

evaluation = WithinSessionEvaluation(
    paradigm=paradigm,
    datasets=datasets,
    suffix="hoda_erp",
    overwrite=True,
    random_state=42,
    n_jobs=5,
)

<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_types is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_channels_regexp is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.channel_type is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.


To use the get_shape_from_baseconcar, InputShapeSetterEEG, BraindecodeDatasetLoaderyou need to install `braindecode`.`pip install braindecode` or Please refer to `https://braindecode.org`.


/usr/local/lib/python3.10/dist-packages/moabb/pipelines/__init__.py:26: ModuleNotFoundError: Tensorflow is not installed. You won't be able to use these MOABB pipelines if you attempt to do so.
  warn(


In [3]:
from sklearn.pipeline import make_pipeline, Pipeline
from hoda.hoda import HODA, BTTDA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from hoda.classification import Vectorize
from hoda.classification import SelectFweAtLeastOne
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import FunctionTransformer
import numpy as np
from hoda.tensorize import hankel_tensor

pipelines = dict()


pipelines['sLDA'] = make_pipeline(
        Vectorize(),
        LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)
"""
pipelines['HODA'] = GridSearchCV(
    Pipeline([
        ('hoda', HODA(
            max_iter=256,
            tol=1e-8,
            init ='svd',
            shrinkage='lw',
            toeplitz=(1,),
            obj='rt',
            solver='lanczos',
            taper=False,
            keep_train_info=False,
            verbose=False,
        )),
        ('vec', Vectorize()),
        ('lda', LinearDiscriminantAnalysis())
    ]),
    dict(hoda__rank=list(range(1,8+1))),
    scoring='roc_auc',
)
"""
"""
pipelines['HODA_aic'] = Pipeline([
    ('bttda', BTTDA(
        max_blocks=1,
        info_crit='aic',
        hoda_params=dict(
            rank=None,
            max_iter=256,
            tol=1e-12,
            init ='svd',
            shrinkage='lw',
            toeplitz=None,
            obj='rt',
            solver='lanczos',
            taper=False,
            keep_train_info=False,
            verbose=False,
        ),
        verbose=False,
        keep_train_info=False,
    )),
    ('vec', Vectorize()),
    ('lda', LinearDiscriminantAnalysis())
])

pipelines['BTTDA_aic'] = Pipeline([
    ('bttda', BTTDA(
        max_blocks=8,
        info_crit='aic',
        hoda_params=dict(
            rank=None,
            max_iter=256,
            tol=1e-12,
            init ='svd',
            shrinkage='lw',
            toeplitz=None,
            obj='rt',
            solver='lanczos',
            taper=False,
            keep_train_info=False,
            verbose=False,
        ),
        verbose=False,
        keep_train_info=False,
    )),
    ('vec', Vectorize()),
    ('lda', LinearDiscriminantAnalysis())
])

"""
pipelines['BTTDA_aic_16'] = Pipeline([
    ('bttda', BTTDA(
        max_blocks=16,
        info_crit='aic',
        hoda_params=dict(
            rank=None,
            max_iter=256,
            tol=1e-12,
            init ='svd',
            shrinkage='lw',
            toeplitz=None,
            obj='rt',
            solver='lanczos',
            taper=False,
            keep_train_info=False,
            verbose=False,
        ),
        verbose=False,
        keep_train_info=False,
    )),
    ('vec', Vectorize()),
    ('lda', LinearDiscriminantAnalysis())
])



In [ ]:
#import warnings
#warnings.filterwarnings("ignore")

results = evaluation.process(pipelines)

BNCI2014-008-WithinSession:   0%|                                                               | 0/8 [00:00<?, ?it/s]

No hdf5_path provided, models will not be saved.


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:312: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))


No hdf5_path provided, models will not be saved.
((5, 5), (4, 4), (2, 2), (4, 4), (1, 1), (1, 1), (1, 1), (2, 2), (1, 1), (1, 1), (3, 3), (2, 2), (3, 3), (2, 2), (2, 2), (1, 1))
((3, 3), (2, 2), (1, 1), (1, 1), (2, 2), (1, 1), (1, 1), (5, 5), (1, 1), (1, 1), (5, 5), (5, 5), (1, 1), (1, 1), (1, 1), (1, 1))
((5, 5), (5, 5), (2, 2), (1, 1), (1, 1), (1, 1), (1, 1), (1, 1), (6, 6), (5, 5), (1, 1), (2, 2), (1, 1), (2, 2), (3, 3), (6, 6))
((5, 5), (5, 5), (2, 2), (7, 7), (2, 2), (2, 2), (2, 2), (7, 7), (2, 2), (1, 1), (3, 3), (1, 1), (1, 1), (1, 1), (1, 1), (1, 1))
((1, 1), (1, 1), (1, 1), (5, 5), (5, 5), (2, 2), (3, 3), (4, 4), (3, 3), (7, 7), (6, 6), (4, 4), (5, 5), (4, 4), (1, 1), (3, 3))


BNCI2014-008-WithinSession:  12%|██████▊                                               | 1/8 [03:25<23:57, 205.38s/it]

...

No hdf5_path provided, models will not be saved.
((3, 3), (2, 2), (1, 1), (2, 2), (6, 6), (1, 1), (5, 5), (1, 1), (2, 2), (1, 1), (2, 2), (5, 5), (1, 1), (3, 3), (4, 4), (1, 1))
((5, 5), (5, 5), (3, 3), (8, 8), (4, 4), (1, 1), (1, 1), (1, 1), (1, 1), (1, 1), (1, 1), (1, 1), (6, 6), (5, 5), (1, 1), (1, 1))
((8, 8), (4, 4), (2, 2), (4, 4), (5, 5), (2, 2), (1, 1), (2, 2), (1, 1), (6, 6), (2, 2), (4, 4), (7, 7), (2, 2), (1, 1), (1, 1))
((5, 5), (2, 2), (4, 4), (3, 3), (4, 4), (1, 1), (3, 3), (4, 4), (1, 1), (1, 1), (5, 5), (1, 1), (1, 1), (5, 5), (1, 1), (5, 5))
((5, 5), (2, 2), (5, 5), (4, 4), (5, 5), (5, 5), (4, 4), (2, 2), (4, 4), (2, 2), (1, 1), (1, 1), (2, 2), (2, 2), (2, 2), (3, 3))


BNCI2014-008-WithinSession:  88%|███████████████████████████████████████████████▎      | 7/8 [22:12<03:20, 200.70s/it]

No hdf5_path provided, models will not be saved.


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:312: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))


No hdf5_path provided, models will not be saved.


In [ ]:
results

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

order = results.groupby('pipeline')
order = order.score.aggregate('mean')
order = order.sort_values()

sns.barplot(
     data=results,
    y="score", x="dataset", hue="pipeline", hue_order=order.index,
)
plt.ylim([.5,1])

In [ ]:
from moabb.analysis.meta_analysis import compute_dataset_statistics, find_significant_differences
from moabb.analysis.plotting import summary_plot
import matplotlib.pyplot as plt

stats = compute_dataset_statistics(results)
P, T = find_significant_differences(stats)
_ = summary_plot(P, T)

In [ ]:
from moabb.analysis.plotting import meta_analysis_plot, paired_plot
_ = meta_analysis_plot(stats, 'HODA_aic', 'BTTDA_aic')
_  = paired_plot(results, 'HODA_aic', 'BTTDA_aic')